# HepSetReg starter notebook

Minimal setup to get `hepsetreg`'s dataset and backbone running in Colab -- enough to load real data and run one batch through the existing model, so you have something working to copy and extend for a new backbone (e.g. a tokenized one).

No training loop/PyTorch Lightning here on purpose -- just the dataset and the backbone, the two pieces you actually need to start from.

## 1. Install `hepsetreg`

In [ ]:
!pip install -q "git+https://github.com/els285/HepSetReg.git@msc_minimal"

## 2. Download the data

Same file `setup_bsc` downloads on the HPC cluster -- just the validation split here (~70MB), since that's enough to explore the data and get a batch running. Download the train file the same way (just swap the filename below) once you're ready to actually train something.

In [ ]:
import os

os.makedirs("datasets", exist_ok=True)
VAL_FILE = "datasets/ttbar-2L_val_051026.parquet"

if not os.path.exists(VAL_FILE):
    !curl -L -o {VAL_FILE} "https://cernbox.cern.ch/s/Pd5y969ajhIi4b3/download?files=ttbar-2L_val_051026.parquet"

## 3. Features

The same field names `configs/run_regression.yaml` uses on the HPC branches -- global/jet/lepton inputs, plus the regression target.

In [ ]:
features = {
    "global": ["global_njet", "global_nelectron", "global_nmuon", "global_nbtagged", "global_met_met", "global_met_phi"],
    "jet": ["jet_pt", "jet_eta", "jet_phi", "jet_e", "jet_m", "jet_px", "jet_py", "jet_pz", "jet_bTag"],
    "lepton": ["pt", "eta", "phi", "e", "m", "px", "py", "pz", "charge", "is_electron"],
    "target": ["ttbar_mass"],
}

## 4. Load one batch

`load_arrays` reads the parquet file's columns straight into (global, jet, lepton, target) arrays -- sliced down to `N_EVENTS` here so this stays quick instead of z-scoring/padding all ~260k events in the file just to look at one batch. `EventDataset` then z-scores every feature and pads the jets to a fixed size per event; `_identity_collate` is a no-op collate function -- `EventDataset` already returns a fully-batched dict, so there's nothing left for the `DataLoader` to collate.

In [ ]:
from torch.utils.data import DataLoader

from hepsetreg.datasets.EventDatasetBase import EventDataset, load_arrays, _identity_collate

N_EVENTS = 10_000  # plenty for exploring the data/model -- raise this once you actually need more

global_arr, jet_arr, lepton_arr, target_arr = load_arrays(VAL_FILE, features)
dataset = EventDataset(
    (global_arr[:N_EVENTS], jet_arr[:N_EVENTS], lepton_arr[:N_EVENTS], target_arr[:N_EVENTS]),
    features=features,
)
loader = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=_identity_collate)
batch = next(iter(loader))

for key, value in batch.items():
    print(f"{key:12s} {tuple(value.shape)} {value.dtype}")

## 5. Run it through the backbone

`TransformerBackbone` turns the global/jet/lepton tokens into one pooled per-event embedding; `RegressionHead` maps that down to the target. **This is the model to copy and adapt for a tokenized approach** -- see `hepsetreg/backbones/backbone.py` for the full implementation.

In [ ]:
import torch

from hepsetreg.backbones.backbone import TransformerBackbone
from hepsetreg.backbones.head import EventRegressor, RegressionHead

backbone = TransformerBackbone(
    n_global_features=len(features["global"]),
    n_jet_features=len(features["jet"]),
    n_lepton_features=len(features["lepton"]),
    d_model=64, nhead=4, num_layers=2,
)
head = RegressionHead(d_model=64, output_dim=len(features["target"]))
model = EventRegressor(backbone, head)

with torch.no_grad():
    pred = model(batch)

print("prediction shape:", tuple(pred.shape))  # (batch_size, len(features["target"]))
print(pred)

## Where to go from here

- `hepsetreg.backbones.backbone.TransformerBackbone` -- the model to copy/extend for a tokenized version. It takes a `batch` dict with keys `global`, `jet`, `jet_mask`, `lepton`, `lepton_mask` (exactly what `EventDataset` produces above) and returns one pooled embedding per event.
- `hepsetreg.backbones.head.RegressionHead` -- the output head mapping that embedding to the target. You usually won't need to change this.
- `hepsetreg.datasets.EventDatasetBase.EventDataset` -- already gives you z-scored, padded tensors; a tokenized backbone can consume the exact same `batch` dict used above.

Once your new backbone works like this, see `src/hepsetreg/runs/run_regression.py` on the HPC branches for how training/prediction (with PyTorch Lightning) is wired up around it.